<a href="https://colab.research.google.com/github/yasminkadhar0321-dot/attendance_system/blob/main/colabs/sampling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sampling

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-deepmind/gemma/blob/main/colabs/sampling.ipynb)

Example on how to load a Gemma model and run inference on it.

The Gemma library has 3 ways to prompt a model:

* `gm.text.ChatSampler`: Easiest to use, simply talk to the model and get answer. Support multi-turns conversations out-of-the-box.
* `gm.text.Sampler`: Lower level, but give more control. The chat state has to be manually handeled for multi-turn.
* `model.apply`: Directly call the model, only predict a single token.

In [4]:
!pip install -q gemma

In [5]:
# Common imports
import os
import jax
import jax.numpy as jnp

# Gemma imports
from gemma import gm

/usr/local/lib/python3.13/dist-packages/jaxlib/plugin_support.py:91: RuntimeWarning: JAX plugin jax_cuda13_plugin version 0.11.1 is installed, but it is not compatible with the installed jaxlib version 0.10.2, so it will not be used.
  warnings.warn(


By default, Jax do not utilize the full GPU memory, but this can be overwritten. See [GPU memory allocation](https://docs.jax.dev/en/latest/gpu_memory_allocation.html):

In [6]:
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

Load the model and the params. Here we load the instruction-tuned version of the model.

In [7]:
model = gm.nn.Gemma3_4B()

params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_4B_IT)

In [8]:

!pip install -q "jax==0.10.2" "jaxlib==0.10.2"


## Multi-turns conversations

The easiest way to chat with Gemma is to use the `gm.text.ChatSampler`. It hides the boilerplate of the conversation cache, as well as the `<start_of_turn>` / `<end_of_turn>` tokens used to format the conversation.

Here, we set `multi_turn=True` when creating `gm.text.ChatSampler` (by default, the `ChatSampler` start a new conversation every time).

In multi-turn mode, you can erase the previous conversation state, by passing `chatbot.chat(..., multi_turn=False)`.

In [9]:
sampler = gm.text.ChatSampler(
    model=model,
    params=params,
    multi_turn=True,
    print_stream=True,  # Print output as it is generated.
)

turn0 = sampler.chat('Share one methapore linking "shadow" and "laughter".')

In [10]:
turn1 = sampler.chat('Expand it in a haiku.')

Note: By default (`multi_turn=False`), the conversation state is reset everytime, but you can still continue the previous conversation by passing `sampler.chat(..., multi_turn=True)`

By default, greedy decoding is used. You can pass a custom `sampling=` method as kwargs:

* `gm.text.Greedy()`: (default) Greedy decoding
* `gm.text.RandomSampling()`: Simple random sampling with temperature, for more variety

## Sample a prompt

For more control, we also provide a `gm.text.Sampler` which still perform efficient sampling (with kv-caching, early stopping,...).

Prompting the sampler require to correctly add format the prompt with the `<start_of_turn>` / `<end_of_turn>` tokens (see the custom token section doc on [tokenizer](https://gemma-llm.readthedocs.io/en/latest/tokenizer.html)).

In [18]:
sampler = gm.text.Sampler(
    model=model,
    params=params,
)

prompt = """<start_of_turn>user
Explain photosynthesis in simple Tamil for a school student.<end_of_turn>
<start_of_turn>model
"""

out = sampler.sample(prompt, max_new_tokens=1000)
print(out)

Okay, let's explain photosynthesis in simple Tamil for a school student!

**Photosynthesis - ஒளிச்சேர்க்கை (Oli Cherkkai)**

சரி, ஒளிச்சேர்க்கைன்னா என்னன்னு பார்ப்போம். இது செடி, கொடி, மரங்கள் எப்படி சாப்பிடுதுன்னு தெரிஞ்சுக்கறதுக்கான ஒரு வழி.

**What is Photosynthesis?**

* **செடிங்க (Sediyanggal) சாப்பிடுறது எப்படி?** (Sediyanggal sappiduru eppadi?) - How do plants eat?
   Normally, we eat food. But plants don't eat like us. They make their own food!

* **சூரியன் (Sooriyan) ஒளி (Oli) முக்கியம் (Mukkiyum).** (Sooriyan oli mukkiyum) - Sunlight is important.
   The sun gives plants energy.

* **காற்று (Kathru) - கார்பன் டை ஆக்சைடு (Carbon Dai Aakside).** (Kathru - Carbon Dai Aakside) - Air - Carbon Dioxide.
   Plants take in a gas called carbon dioxide from the air.  It's like a gas we breathe out.

* **தண்ணீர் (Thanneer) - வேர்கள் (Veergal).** (Thanneer - Veergal) - Water - Roots.
   Plants drink water through their roots.

* **பசுமை (Pasumai) - குளோரோஃபில் (Chlorophil).** (Pasumai - C

## Use the model directly

Here's an example of predicting a single token, directly calling the model.

The model input expectes encoded tokens. For this, we first need to encode the prompt with our tokenizer. See our [tokenizer](https://gemma-llm.readthedocs.io/en/latest/tokenizer.html) documentation for more information on using the tokenizer.

In [12]:
tokenizer = gm.text.Gemma3Tokenizer()

Note: When encoding the prompt, don't forget to add the beginning-of-string token with `add_bos=True`. All prompts feed to the model should start by this token.

In [13]:
prompt = tokenizer.encode('One word to describe Paris: \n\n', add_bos=True)
prompt = jnp.asarray(prompt)

We then can call the model, and get the predicted logits.

In [14]:
# Run the model
out = model.apply(
    {'params': params},
    tokens=prompt,
    return_last_only=True,  # Only predict the last token
)


# Sample a token from the predicted logits
next_token = jax.random.categorical(
    jax.random.key(1),
    out.logits
)
tokenizer.decode(next_token)

'Romantic'

You can also display the next token probability.

In [15]:
tokenizer.plot_logits(out.logits)

In [17]:

# Test Gemma with an educational prompt

prompt = "Explain photosynthesis in simple Tamil for a school student."

answer = sampler.chat(prompt)

print("Prompt:", prompt)
print("\nGemma's Answer:")
print(answer)



AttributeError: 'Sampler' object has no attribute 'chat'

## Next steps

* See our [multimodal](https://gemma-llm.readthedocs.io/en/latest/multimodal.html) example to query the model with images.
* See our [finetuning](https://gemma-llm.readthedocs.io/en/latest/finetuning.html) example to train Gemma on your custom task.
* See our [tool use](https://gemma-llm.readthedocs.io/en/latest/tool_use.html) tutorial to extend Gemma with external tools.
